# Implement Residual (Skip) Connection from Scratch
### Problem Statement
Residual connections (skip connections), introduced in [He et al. 2015](https://arxiv.org/abs/1512.03385) for ResNets, are a critical component of the Transformer. The idea is simple but powerful: instead of learning `F(x)`, learn `F(x) + x`. This allows gradients to flow directly through the network, enabling training of very deep models.

In Transformers, residual connections wrap both the attention sublayer and the feed-forward sublayer. Combined with normalization, there are two common patterns:

- **Post-Norm** (original Transformer): `LayerNorm(x + Sublayer(x))`
- **Pre-Norm** (GPT-2, Llama, modern LLMs): `x + Sublayer(Norm(x))`

Your goal is to implement both patterns and understand why Pre-Norm is preferred for deep models.

---

### Requirements

1. **Post-Norm Residual Block**
   - `output = LayerNorm(x + sublayer(x))`
   - Normalization happens **after** the residual addition.

2. **Pre-Norm Residual Block**
   - `output = x + sublayer(Norm(x))`
   - Normalization happens **before** the sublayer, residual path is clean.

3. **Demonstrate Gradient Flow**
   - Show that residual connections preserve gradient magnitude through many layers.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ The sublayer can be any `nn.Module` (attention, FFN, etc.).
- ✅ Both Pre-Norm and Post-Norm must be implemented.

---

<details>
  <summary>💡 Hint</summary>

  - The residual connection is literally just `x + f(x)` — the key design choice is where normalization goes.
  - Pre-Norm leaves the residual stream "clean" — the identity path `x` has no transformations applied to it.
  - Optional dropout is often applied to `sublayer(x)` before adding back to `x`.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class PostNormResidual(nn.Module):
    """
    Post-Norm residual connection: LayerNorm(x + Sublayer(x))
    Used in the original Transformer (Vaswani et al. 2017).
    """
    def __init__(self, d_model: int, sublayer: nn.Module, dropout: float = 0.1):
        super().__init__()
        self.sublayer = sublayer
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x + self.dropout(self.sublayer(x)))

In [ ]:
class PreNormResidual(nn.Module):
    """
    Pre-Norm residual connection: x + Sublayer(Norm(x))
    Used in GPT-2, Llama, and most modern LLMs.
    """
    def __init__(self, d_model: int, sublayer: nn.Module, dropout: float = 0.1):
        super().__init__()
        self.sublayer = sublayer
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.dropout(self.sublayer(self.norm(x)))

## 📚 Why Residual Connections Matter

### The Gradient Flow Problem

Without residual connections, gradients must pass through every transformation layer during backpropagation. In a deep network with layers `f1, f2, ..., fN`, the gradient at the input is:

```
∂L/∂x = ∂L/∂fN · ∂fN/∂fN-1 · ... · ∂f2/∂f1 · ∂f1/∂x
```

This chain of multiplications causes gradients to either **vanish** (if factors < 1) or **explode** (if factors > 1).

### How Residual Connections Help

With `y = x + f(x)`, the gradient becomes:

```
∂y/∂x = 1 + ∂f(x)/∂x
```

The **+1 term** guarantees the gradient is at least 1, preventing vanishing gradients. Through N layers:

```
∂L/∂x = ∂L/∂yN · (1 + ∂fN/∂x) · (1 + ∂fN-1/∂x) · ... · (1 + ∂f1/∂x)
```

Every term in the product is at least 1, so gradients can flow freely.

### Pre-Norm vs Post-Norm

```
Post-Norm: LayerNorm(x + Sublayer(x))
  - Norm is on the residual path → gradients must pass through norm
  - Harder to train for very deep models
  - May need learning rate warmup

Pre-Norm:  x + Sublayer(Norm(x))
  - Residual path is completely clean (just addition)
  - Gradients flow through identity path unobstructed
  - More stable training, especially for deep models
  - Used by GPT-2, GPT-3, Llama, Mistral, etc.
```

### The Residual Stream View

Modern interpretability research views the transformer as a **residual stream** — information flows through the identity path, and each attention/FFN layer reads from and writes to this stream:

```
x₀ ──────────────────────────────────────────────> x_final
    ↘ Norm → Attn ↗  ↘ Norm → FFN ↗  ↘ Norm → Attn ↗
       Layer 1            Layer 1          Layer 2
```

In [ ]:
# Test both patterns
torch.manual_seed(42)
d_model = 8
x = torch.randn(3, 4, d_model)

# Use a simple linear layer as the sublayer
sublayer = nn.Linear(d_model, d_model)

post_norm = PostNormResidual(d_model, sublayer, dropout=0.0)
pre_norm = PreNormResidual(d_model, sublayer, dropout=0.0)

out_post = post_norm(x)
out_pre = pre_norm(x)

print(f"Input shape:      {x.shape}")
print(f"Post-Norm output: {out_post.shape}")
print(f"Pre-Norm output:  {out_pre.shape}")
assert out_post.shape == x.shape
assert out_pre.shape == x.shape
print("\n✅ Both residual connection patterns work correctly.")

In [ ]:
# Bonus: Gradient flow comparison
# Stack many layers and check gradient magnitude at the input

def check_gradient_flow(num_layers, use_residual=True):
    """Stack num_layers linear layers and measure gradient at input."""
    d = 64
    x = torch.randn(1, d, requires_grad=True)
    out = x
    layers = [nn.Linear(d, d, bias=False) for _ in range(num_layers)]
    
    for layer in layers:
        if use_residual:
            out = out + layer(out)  # residual
        else:
            out = layer(out)         # no residual
    
    loss = out.sum()
    loss.backward()
    return x.grad.norm().item()

for n in [5, 20, 50]:
    grad_residual = check_gradient_flow(n, use_residual=True)
    grad_plain = check_gradient_flow(n, use_residual=False)
    print(f"Layers={n:2d} | With residual: {grad_residual:.4f} | Without: {grad_plain:.6f}")